# Published Darcy curves: archived results

This notebook displays archived **pyMHM** results and digitized curves from
**Figure 5, p. 117** of [Harder, Paredes and Valentin (2013)](https://doi.org/10.1016/j.jcp.2013.03.019).
The problem is the published cosine-2π Neumann problem, distinct from the
cosine-π Dirichlet example.

The recorded classical RT0 flux and its mean-preserving quadratic potential match the
five digitized levels within **0.20%**, under the declared 1% extraction
allowance. The distinct P1 results are retained: their flux and projected-mean
errors differ. Agreement with digitized curves does not identify the original
code, mesh connectivity or source-lift implementation.

All values, acceptance flags and figures below were recorded previously.
This notebook reads those artifacts without running a finite element solve or
recomputing the published-curve comparison. The formulation and qualifications
are described in [the reproduction case page](https://ipes-lncc.github.io/pymhm/cases/reproduction/).

The full-source analytical MHM of equation (42) is a separate formulation. Notebook `43_analytical_darcy.ipynb` reports its six-level comparison; its coarse pressure does not coincide with classical RT0 for this source.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/ccc91a220552667782119297da969b00c959dd1809cdde8d19fbcb3d7d73e617/11_published_darcy_2013-companion.zip"
COMPANION_SHA256 = "ccc91a220552667782119297da969b00c959dd1809cdde8d19fbcb3d7d73e617"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/11_published_darcy_2013.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
from pathlib import Path
import csv
import json
from IPython.display import Image, JSON, display


with (root / "examples/results/published/harder2013_figure5.csv").open(newline="") as stream:
    published = list(csv.DictReader(stream))
records = {
    "primal": json.loads((root / "examples/results/darcy_2013_comparison.json").read_text()),
    "mixed": json.loads((root / "examples/results/darcy_2013_mixed_comparison.json").read_text()),
}

## Declared problem and grids

On the unit square, $p=\cos(2\pi x)\cos(2\pi y)$, $K=I$, $f=8\pi^2p$, $q=-\nabla p$, $q\cdot n=0$ on the whole boundary, and $\int_\Omega p=0$.

The candidate grid splits every square along its lower-left to upper-right diagonal. It has $2n^2$ macro triangles, one local triangle per macrocell, and degree-zero traces. Its spacing is $s=1/n$ and its geometric maximum diameter is $\sqrt2/n$. Figure 5's labels are paired with $s$; the paper's diameter/grid-count convention remains unresolved.

Assembly uses Duffy order 10 and error integration order 12. A stored order-12/order-14 check confirms that quadrature changes are negligible compared with the figure-extraction tolerance.

In [ ]:
display(JSON({"source": "Harder et al. (2013), digitized Figure 5", "values": published}))
for method, record in records.items():
    display(JSON({
        "code": "pyMHM",
        "formulation": method,
        "local_space": record["local_space"],
        "recorded_n4_result": record["results"][0],
        "recorded_quadrature_check": record["quadrature_check"],
    }))

## Why the updated pressure is quadratic for RT0

For one triangle, write $q_h(x)=q_h(c_T)+b_T(x-c_T)$, where $b_T=(\nabla\cdot q_h)/2$. Its mean-preserving potential is

$$p_T^\star=p_{0,T}-q_h(c_T)\cdot(x-c_T)-\frac{b_T}{2}\left(|x-c_T|^2-\overline{|x-c_T|^2}_T\right).$$

Thus $-\nabla p_T^\star=q_h$ and $\Pi_0p_T^\star=p_{0,T}$. This radial quadratic form is justified by equation (40) in the article. The native mixed pressure is still P0; the updated-pressure curve uses $p_T^\star$ explicitly.

The reconstructed potential has $-\Delta p_T^\star=\Pi_0f$. It does not restore the exact source lift for $f-\Pi_0f$. The article's source-dependent load correction and the original code path are unresolved even though the plotted values agree.

In [ ]:
for method, folder in (("mixed", "mixed/"), ("primal", "")):
    record = records[method]
    display(JSON({
        "code": "pyMHM",
        "formulation": method,
        "recorded_comparison_status": record["comparison_status"],
        "figure5_norm_mapping": record["figure5_comparison_fields"],
        "recorded_acceptance_by_norm": record["all_points_within_digitization_tolerance"],
    }))
    path = root / f"docs/figures/reproduction/{folder}darcy-2013-comparison.png"
    display(Image(filename=str(path), width=1000))

## Recorded rates and acceptance

The four norms are updated-pressure L2, physical-flux L2, coarse-pressure L2
and projected-coarse-pressure L2. All are absolute norms. Each archived study
contains five points; no level is recomputed in this notebook.

The stored final observed orders for RT0 with quadratic pressure recovery are
approximately 2, 1, 1 and 2. Similar slopes also appear for P1, but its error
constants differ from Figure 5. Numerical rates alone do not establish
reproduction. The mapping above distinguishes native mixed P0 pressure from
the recovered pressure used in the published updated-pressure curve.

In [ ]:
for method, record in records.items():
    last = record["results"][-1]
    display(JSON({
        "code": "pyMHM",
        "formulation": method,
        "final_macro_resolution": last["n"],
        "recorded_orders": {key: value for key, value in last.items() if key.endswith("_eoc")},
        "recorded_acceptance_by_norm": record["all_points_within_digitization_tolerance"],
    }))